# DES Y3 Shear Maps for NaMaster

This notebook documents and exercises the DES Y3 shear-map product in this transfer folder.  The heavy map-building logic lives in `scripts/prepare_des_y3_shear_maps.py`; the cells below show how to rebuild the product, inspect the HDF5 schema, visualize the maps, and set up NaMaster fields for shear auto, shear-tSZ, and shear-galaxy harmonic-space measurements.

References used for choices here:

- DES Y3 harmonic-space paper: `/global/cfs/cdirs/lsst/www/shivamp/DESI/2203.07128v1.pdf`, https://arxiv.org/abs/2203.07128, https://academic.oup.com/mnras/article/515/2/1942/6625643
- Source DES shear processing notebook: `/global/cfs/cdirs/lsst/www/shivamp/DESI/ACTxDES_measurements.ipynb`
- NaMaster spin-2 convention docs: https://namaster.readthedocs.io/en/latest/api/pymaster.field.html

The DES Y3 paper uses weighted tomographic shear maps at HEALPix `nside=1024`, weighted galaxy-count masks, no E/B purification, and 32 square-root-spaced bandpowers from `ell=8` to `ell=2048`.  This transfer folder contains that fiducial-resolution product plus a separate `nside=4096` high-resolution product for small-scale/high-ell tests.

In [ ]:
from pathlib import Path
import json
import subprocess

import h5py
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits

try:
    import pymaster as nmt
except ImportError:
    nmt = None
    print('pymaster/NaMaster is not importable in this kernel; measurement cells will need that package.')

def find_transfer_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'manifest.json').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the act_desi_ksz_transfer package.')


OUTDIR = find_transfer_root()
SCRIPT = OUTDIR / 'scripts/prepare_des_y3_shear_maps.py'
SHEAR_H5_1024 = OUTDIR / 'data/des_y3_shear_maps/des_y3_metacal_shear_maps_nside1024.h5'
SHEAR_H5_4096 = OUTDIR / 'data/des_y3_shear_maps/des_y3_metacal_shear_maps_nside4096.h5'
SHEAR_H5 = SHEAR_H5_1024  # switch to SHEAR_H5_4096 for high-ell tests
TSZ_H5 = OUTDIR / 'data/act_dr6_tsz_compton_y/act_dr6_nilc_compton_y_deproj_cib_cibdbeta_1p7_10p7_transfer.h5'
DESI_ALL_H5 = OUTDIR / 'data/desi_dr10_extended_velocity_catalogs/desi_dr10_extended_all_pz_compact.h5'
DESI_RANDOM_FITS = OUTDIR / 'data/desi_dr10_imaging_randoms/randoms-1-0.fits'
DOC = OUTDIR / 'docs/DES_Y3_SHEAR_MAPS.md'

print('default shear file:', SHEAR_H5)
print('high-res shear file:', SHEAR_H5_4096)
print(DOC)

## Optional Rebuild

Both products have already been built.  Set `RUN_REBUILD = True` only if you want to regenerate them from the processed DES shear pickle.  The script defaults to checking/writing both `nside=1024` and `nside=4096`; use `--nside 4096` if you only want the high-resolution file.

In [ ]:
RUN_REBUILD = False

if RUN_REBUILD:
    cmd = [
        '/global/homes/s/spandey/.conda/envs/myenv_conda/bin/python',
        str(SCRIPT),
        '--force',
    ]
    env = {
        'MPLCONFIGDIR': '/tmp/act_desi_ksz_mplconfig',
        'XDG_CACHE_HOME': '/tmp/act_desi_ksz_xdgcache',
    }
    subprocess.run(cmd, check=True, env={**__import__('os').environ, **env}, cwd=OUTDIR)
else:
    print('Skipping rebuild. Existing products:', SHEAR_H5_1024, SHEAR_H5_4096)

## Inspect the HDF5 Product

In [ ]:
with h5py.File(SHEAR_H5, 'r') as h5:
    print('top-level groups:', list(h5.keys()))
    print('nside:', h5.attrs['nside'])
    print('npix:', h5.attrs['npix'])
    print('map groups:', list(h5['maps'].keys()))
    print('tomo0 datasets:', list(h5['maps/tomo0'].keys()))
    print('bandpower left edges:', h5['bandpowers/ell_left'][:5], '...')
    print('bandpower right edges:', h5['bandpowers/ell_right'][:5], '...')
    summary = h5['summary/tomographic_summary_table'][:]

summary

In [ ]:
with h5py.File(SHEAR_H5, 'r') as h5:
    for row in h5['summary/tomographic_summary_table'][:]:
        print(
            f"tomo {row['tomo']}: "
            f"N={row['n_sources']:,}, "
            f"area={row['area_deg2']:.2f} deg^2, "
            f"mean count={row['mean_count']:.3f}, "
            f"n_eff={row['n_eff_arcmin2']:.3f} arcmin^-2"
        )

## Quicklook Figures

The script saved Mollweide views for the weighted masks and both shear components in all four tomographic bins.

In [ ]:
from IPython.display import Image, display

for name in [
    'des_y3_shear_tomo1_mask_weight_nside1024.png',
    'des_y3_shear_tomo1_gamma1_nside1024.png',
    'des_y3_shear_tomo1_gamma2_nside1024.png',
    'des_y3_shear_mean_counts_nside1024.png',
    'des_y3_shear_neff_nside1024.png',
]:
    path = OUTDIR / 'quicklook_figures' / name
    print(path.name)
    display(Image(filename=str(path)))

## Shear Field Helpers

Use `gamma2_namaster`, not raw `gamma2`, when constructing NaMaster spin-2 fields.  This implements the sign flip required when a galaxy ellipticity catalog follows the usual IAU convention while NaMaster uses the HEALPix/CMB spin convention.

In [ ]:
def require_namaster():
    if nmt is None:
        raise ImportError('Install/import pymaster before running NaMaster cells.')


def make_bins(h5):
    require_namaster()
    return nmt.NmtBin.from_edges(h5['bandpowers/ell_left'][:], h5['bandpowers/ell_right'][:])


def make_shear_field(h5, tomo, mask_name='mask_weight_raw'):
    require_namaster()
    g = h5[f'maps/tomo{tomo}']
    mask = g[mask_name][:]
    gamma1 = g['gamma1'][:]
    gamma2 = g['gamma2_namaster'][:]
    return nmt.NmtField(mask, [gamma1, gamma2], spin=2, purify_e=False, purify_b=False)


def shape_noise_coupled_like(h5, tomo, coupled, mask_name='mask_weight_raw'):
    g = h5[f'maps/tomo{tomo}']
    attr_by_mask = {
        'mask_weight_raw': 'shape_noise_pseudo_cl_raw_weight_mask',
        'mask_weight': 'shape_noise_pseudo_cl_normalized_weight_mask',
        'mask_binary': 'shape_noise_pseudo_cl_binary_mask',
    }
    n_ell = g.attrs[attr_by_mask[mask_name]]
    noise = np.zeros_like(coupled)
    noise[0, :] = n_ell  # EE
    noise[3, :] = n_ell  # BB
    return noise

## Shear Auto Example

This computes a single tomo-1 auto spectrum when `RUN_NAMASTER_EXAMPLE` is set to `True`.  Same-bin auto spectra subtract the stored shape-noise pseudo-spectrum before decoupling.  Cross-bin shear spectra should skip the noise subtraction.

In [ ]:
RUN_NAMASTER_EXAMPLE = False

if RUN_NAMASTER_EXAMPLE:
    with h5py.File(SHEAR_H5, 'r') as h5:
        bins = make_bins(h5)
        field = make_shear_field(h5, tomo=0, mask_name='mask_weight_raw')
        workspace = nmt.NmtWorkspace()
        workspace.compute_coupling_matrix(field, field, bins)
        coupled = nmt.compute_coupled_cell(field, field)
        noise = shape_noise_coupled_like(h5, tomo=0, coupled=coupled, mask_name='mask_weight_raw')
        cl_decoupled = workspace.decouple_cell(coupled - noise)
        ell_eff = bins.get_effective_ells()

    plt.figure(figsize=(7, 4.5))
    plt.loglog(ell_eff, np.abs(cl_decoupled[0]), marker='o', label='EE')
    plt.loglog(ell_eff, np.abs(cl_decoupled[3]), marker='s', label='BB')
    plt.xlabel('ell')
    plt.ylabel('|C_ell|')
    plt.legend()
    plt.tight_layout()
else:
    print('Set RUN_NAMASTER_EXAMPLE=True to run the shear auto example.')

## Shear x tSZ Setup

The ACT tSZ map in this package is stored as a pixell/enmap CAR map.  For a HEALPix shear cross-spectrum, reproject the tSZ map and mask to the same HEALPix `nside` as the chosen DES shear file, then create a spin-0 field and cross it with each DES spin-2 field.  This cell is written as a ready recipe but is off by default because it reads a large CAR map.

In [ ]:
RUN_TSZ_REPROJECT = False

def read_enmap_from_hdf5(h5, dataset, wcs_attr='map_wcs_header'):
    from astropy.io import fits
    from astropy.wcs import WCS
    from pixell import enmap

    header = fits.Header.fromstring(h5['geometry'].attrs[wcs_attr], sep='\n')
    return enmap.enmap(h5[dataset][:], WCS(header))


if RUN_TSZ_REPROJECT:
    from pixell import reproject

    with h5py.File(SHEAR_H5, 'r') as sh5, h5py.File(TSZ_H5, 'r') as th5:
        nside = int(sh5.attrs['nside'])
        y_car = read_enmap_from_hdf5(th5, 'maps/compton_y', 'map_wcs_header')
        mask_car = read_enmap_from_hdf5(th5, 'masks/footprint_mask', 'footprint_mask_wcs_header')
        y_hp = reproject.map2healpix(y_car, nside=nside, lmax=3*nside-1, method='harm')
        mask_y_hp = reproject.map2healpix(mask_car, nside=nside, lmax=3*nside-1, method='harm')
        mask_y_hp = np.clip(mask_y_hp, 0.0, 1.0)
        field_y = nmt.NmtField(mask_y_hp, [y_hp], spin=0)
        field_shear = make_shear_field(sh5, tomo=0)
        bins = make_bins(sh5)
        workspace = nmt.NmtWorkspace()
        workspace.compute_coupling_matrix(field_y, field_shear, bins)
        cl_y_gamma = workspace.decouple_cell(nmt.compute_coupled_cell(field_y, field_shear))

    print('spin-0 x spin-2 spectra shape:', cl_y_gamma.shape)
else:
    print('Set RUN_TSZ_REPROJECT=True to reproject the tSZ map and run a y x shear example.')

## Shear x DESI Galaxy Setup

The DESI catalogs are stored separately in this transfer directory.  Pixelize any desired DESI selection to the same HEALPix `nside` as the chosen shear file, create a spin-0 overdensity field, and cross it with the DES spin-2 shear field.  NaMaster can use different masks for the two fields.

In [ ]:
def make_desi_random_counts(random_fits, nside, npix, chunk=2_000_000):
    counts = np.zeros(npix, dtype=np.uint32)
    with fits.open(random_fits, memmap=True) as hdul:
        tab = hdul[1].data
        for start in range(0, len(tab), chunk):
            rows = tab[start:start + chunk]
            photsys = np.asarray(rows['PHOTSYS']).astype('U1')
            good = (
                (photsys == 'S')
                & (rows['NOBS_G'] > 0)
                & (rows['NOBS_R'] > 0)
                & (rows['NOBS_Z'] > 0)
                & (rows['MASKBITS'] == 0)
            )
            if not np.any(good):
                continue
            pix = hp.ang2pix(
                nside,
                np.radians(90.0 - np.asarray(rows['DEC'][good], dtype=float)),
                np.radians(np.mod(np.asarray(rows['RA'][good], dtype=float), 360.0)),
            )
            counts += np.bincount(pix, minlength=npix).astype(np.uint32)
    return counts.astype(np.float64)


def make_desi_density_map(catalog_h5, random_fits, nside=1024, pz_bin=None, z_min=None, z_max=None):
    npix = hp.nside2npix(nside)
    with h5py.File(catalog_h5, 'r') as h5:
        ra = h5['catalog/ra_deg'][:]
        dec = h5['catalog/dec_deg'][:]
        z = h5['catalog/z'][:]
        sel = np.isfinite(ra) & np.isfinite(dec)
        if 'pz_bin' in h5['catalog'] and pz_bin is not None:
            sel &= h5['catalog/pz_bin'][:] == pz_bin
        if z_min is not None:
            sel &= z >= z_min
        if z_max is not None:
            sel &= z < z_max

    pix = hp.ang2pix(nside, np.radians(90.0 - dec[sel]), np.radians(np.mod(ra[sel], 360.0)))
    galaxy_counts = np.bincount(pix, minlength=npix).astype(np.float64)
    random_counts = make_desi_random_counts(random_fits, nside, npix)
    mask = random_counts > 0
    alpha = galaxy_counts[mask].sum() / random_counts[mask].sum()
    delta = np.zeros(npix, dtype=np.float32)
    delta[mask] = (galaxy_counts[mask] / (alpha * random_counts[mask]) - 1.0).astype(np.float32)
    mask_weight = np.zeros(npix, dtype=np.float32)
    mask_weight[mask] = (random_counts[mask] / random_counts[mask].mean()).astype(np.float32)
    return delta, mask_weight, galaxy_counts.astype(np.float32), random_counts.astype(np.float32)


RUN_DESI_CROSS_EXAMPLE = False

if RUN_DESI_CROSS_EXAMPLE:
    with h5py.File(SHEAR_H5, 'r') as sh5:
        nside = int(sh5.attrs['nside'])
        delta_g, mask_g, counts, random_counts = make_desi_density_map(DESI_ALL_H5, DESI_RANDOM_FITS, nside=nside, pz_bin=1)
        field_g = nmt.NmtField(mask_g, [delta_g], spin=0)
        field_shear = make_shear_field(sh5, tomo=0)
        bins = make_bins(sh5)
        workspace = nmt.NmtWorkspace()
        workspace.compute_coupling_matrix(field_g, field_shear, bins)
        cl_g_gamma = workspace.decouple_cell(nmt.compute_coupled_cell(field_g, field_shear))
    print('spin-0 x spin-2 spectra shape:', cl_g_gamma.shape)
else:
    print('Set RUN_DESI_CROSS_EXAMPLE=True to run a DESI density x shear example.')

## Transfer Checklist

Copy the whole `act_desi_ksz_transfer` directory for later measurement.  The relevant DES shear files are:

- `data/des_y3_shear_maps/des_y3_metacal_shear_maps_nside1024.h5`
- `data/des_y3_shear_maps/des_y3_metacal_shear_maps_nside4096.h5`
- `scripts/prepare_des_y3_shear_maps.py`
- `prepare_des_y3_shear_maps.ipynb`
- `docs/DES_Y3_SHEAR_MAPS.md`
- `manifest.json`
- `quicklook_figures/des_y3_shear_*.png`

For shear-tSZ measurements also transfer `data/act_dr6_tsz_compton_y/`.  For shear-galaxy measurements also transfer `data/desi_dr10_extended_velocity_catalogs/`.